# 01 Data audit

Row counts, column inventory, unknown share, duplicates, and missingness for every source. Feeds `docs/data_audit.md` (Gate 0). Outputs are aggregates only; no individual addresses are printed.

In [1]:
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from lll import PROCESSED_DIR, sources
from lll.features import line_status

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.3f}".format)
SNAP = "20250414"
FIG = PROCESSED_DIR.parent.parent / "docs" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
from lll.ingest import read_iepa_template, read_il_epa_summary

## 1. Illinois EPA inventory reports (system level)

In [2]:
il = read_il_epa_summary(sources.IL_EPA_SUMMARY)
print(f"rows {len(il):,}  systems {il.FacilityId.nunique():,}  years {il.ReportingYear.min()}-{il.ReportingYear.max()}")
dupes = il.duplicated(["FacilityId", "ReportingYear"], keep=False)
print(f"system-year rows reported more than once: {dupes.sum():,}")
by_year = il.groupby("ReportingYear").agg(
    systems=("FacilityId", "nunique"),
    connections=("TotalServiceConnections", "sum"),
    lead=("ServicesLead", "sum"),
    unknown=("ServicesUnknownMaterial", "sum"),
    galvanized=("ServicesGalvanized", "sum"),
)
by_year["lead_share"] = by_year.lead / by_year.connections
by_year["unknown_share"] = by_year.unknown / by_year.connections
by_year

rows 9,100  systems 1,794  years 2017-2025
system-year rows reported more than once: 12


,systems,connections,lead,unknown,galvanized,lead_share,unknown_share
ReportingYear,,,,,,,
2017,1659,3759199,414895,1527508,67273,0.110,0.406
2018,1748,3841625,686259,1067119,59664,0.179,0.278
2019,1754,3847019,678529,793846,59700,0.176,0.206
2020,1756,3873180,667275,819076,62958,0.172,0.211
2021,136,290487,35689,52106,6343,0.123,0.179
2022,1429,3519377,598680,586232,1384,0.170,0.167
2023,9,8561,500,3397,0,0.058,0.397
2024,602,1622685,250617,406960,0,0.154,0.251
2025,1,338,0,0,0,0.000,0.000


In [3]:
chi = il[il.FacilityId.eq("IL0316000")].sort_values("ReportingYear")
chi[["ReportingYear", "ReportedDate", "TotalServiceConnections", "ServicesLead",
     "ServicesUnknownMaterial", "ServicesGalvanized", "lead_share", "unknown_share"]]

,ReportingYear,ReportedDate,TotalServiceConnections,ServicesLead,ServicesUnknownMaterial,ServicesGalvanized,lead_share,unknown_share
8209,2017,2018-04-12 17:14:22,494155,120010,362843,11298,0.243,0.734
6201,2018,2019-04-15 18:14:05,520673,392614,120760,0,0.754,0.232
4281,2019,2020-04-15 15:37:24,520612,389893,119468,19,0.749,0.229
2400,2020,2021-04-15 17:21:33,522571,387095,118746,42,0.741,0.227
938,2022,2023-04-16 12:45:06,499548,409734,0,0,0.820,0.000
395,2024,2025-04-14 16:12:51,487347,150767,251569,0,0.309,0.516


In [4]:
latest = il.sort_values(["ReportingYear", "ReportedDate"]).groupby("FacilityId").tail(1)
r24 = latest[latest.ReportingYear.eq(2024)]
detail = ["ServicesCopperLeadSolder", "ServicesGalvanized", "ServicesUnknownNotLead",
          "ServicesCopperNonLeadSolder", "ServicesCastDuctileIronTransite", "ServicesPlastic"]
print(f"2024 reports: {len(r24):,}; with any non-lead detail column filled: "
      f"{(r24[detail].fillna(0).sum(axis=1) > 0).sum():,}")
print(f"2024 statewide: lead {r24.ServicesLead.sum():,}, unknown {r24.ServicesUnknownMaterial.sum():,}, "
      f"connections {r24.TotalServiceConnections.sum():,}")
print(f"systems whose lead+unknown exceeds connections: "
      f"{(latest.ServicesLead.fillna(0) + latest.ServicesUnknownMaterial.fillna(0) > latest.TotalServiceConnections).sum()}")

2024 reports: 602; with any non-lead detail column filled: 0
2024 statewide: lead 250,617, unknown 406,960, connections 1,622,685
systems whose lead+unknown exceeds connections: 0


## 2. Chicago line-level inventory (IEPA template, 2025-04-14)

In [5]:
inv, meta = read_iepa_template(sources.CHICAGO_INVENTORY)
print(meta["system_id"], meta["system_name"], meta["snapshot_date"].date())
counted = line_status(inv.utility_side_material, inv.customer_side_material).value_counts()
pd.DataFrame({"header_total": pd.Series(meta["header_totals"]),
              "rows": pd.Series({"total": len(inv), "lead": counted.get("Lead"),
                                  "nonlead": counted.get("Non-Lead"), "grr": counted.get("GRR"),
                                  "unknown": counted.get("Unknown")})})

IL0316000 CHICAGO 2025-04-14


,header_total,rows
total,491705,491705
lead,150864,150864
nonlead,79068,79068
grr,10526,10526
unknown,251247,251247


In [6]:
sides = pd.DataFrame({side: inv[side].value_counts(normalize=True)
                      for side in ["utility_side_material", "customer_side_material",
                                   "reported_line_class", "gooseneck_material"]})
sides

,utility_side_material,customer_side_material,reported_line_class,gooseneck_material
GRR,0.001,0.026,0.021,0.001
Lead,0.103,0.235,0.307,0.103
Non-Lead,0.171,0.188,0.161,0.171
Unknown,0.726,0.552,0.511,0.726


In [7]:
print("derived whole-line status equals reported OverallSL Code:",
      (line_status(inv.utility_side_material, inv.customer_side_material) == inv.reported_line_class).mean())
miss = inv.isna().mean().rename("missing_share").to_frame()
miss[miss.missing_share > 0].sort_values("missing_share")

derived whole-line status equals reported OverallSL Code: 1.0


,missing_share
verification_method,0.496
notification_date,0.951
utility_install_year,0.963
install_year,0.969
replacement_date,0.978
pin,1.000
year_built,1.000
diameter_in,1.000


In [8]:
addr = inv.address.str.upper().str.strip()
print(f"exact duplicate rows: {inv.drop(columns='line_id').duplicated().sum():,}")
print(f"lines sharing an address with another line: {addr.duplicated(keep=False).sum():,} "
      f"({addr.duplicated(keep=False).mean():.2%}); unique addresses {addr.nunique():,}")
snap_year = meta["snapshot_date"].year
checks = {
    "utility install year < 1850": (inv.utility_install_year < 1850).sum(),
    "customer install year < 1850": (inv.install_year < 1850).sum(),
    "install year after snapshot": ((inv.install_year > snap_year) | (inv.utility_install_year > snap_year)).sum(),
    "replacement date after snapshot": (inv.replacement_date > meta["snapshot_date"]).sum(),
    "replacement date before 1990": (inv.replacement_date < "1990-01-01").sum(),
    "replaced flag but side still Lead": (inv.prev_lead_replaced & inv.utility_side_material.eq("Lead")).sum(),
    "classification based on Home Built Date": inv.verification_method.str.contains("Home Built Date", na=False).sum(),
}
pd.Series(checks, name="rows").to_frame()

exact duplicate rows: 2,285


lines sharing an address with another line: 7,470 (1.52%); unique addresses 487,669


,rows
utility install year < 1850,0
customer install year < 1850,0
install year after snapshot,0
replacement date after snapshot,0
replacement date before 1990,0
replaced flag but side still Lead,0
classification based on Home Built Date,5050


In [9]:
inv.verification_method.str.extract(r"^(P[RU]):")[0].value_counts(dropna=False).rename("source prefix")

0
<NA>    243998
PU      128085
PR      119622
Name: source prefix, dtype: int64[pyarrow]

## 3. Assessor parcels and building characteristics

In [10]:
p = sources.chicago_parcels()
print(f"parcel-address rows {len(p):,}, unique PINs {p.pin.nunique():,}, buildings (pin10) {p.pin.str[:10].nunique():,}")
print(f"missing coordinates {p.latitude.isna().sum():,}; missing block GEOID {p.census_block_geoid.isna().sum():,}")
print(p.building_class.astype(str).str[0].value_counts().rename("class major").head(10).to_string())

parcel-address rows 880,789, unique PINs 880,789, buildings (pin10) 604,414
missing coordinates 0; missing block GEOID 1
building_class
2    732250
E     49256
5     48265
1     32359
3     14215
R      2780
9       737
6       482
4       235
7       201


In [11]:
chars = sources.building_characteristics()
print(f"buildings with characteristics {len(chars):,}")
print(chars.year_built.describe().to_string())
raw_yr = pd.concat([pd.read_csv(f, usecols=["pin", "char_yrblt"], dtype=str)
                    for f in (sources.RES_CHARS, sources.CONDO_CHARS)])
raw_min = pd.to_numeric(raw_yr.char_yrblt, errors="coerce").groupby(raw_yr.pin.str[:10]).min()
print("raw buildings with year built < 1830 (set to missing):", int((raw_min < 1830).sum()),
      "| earliest raw year:", int(raw_min.min()), "| after 2026:", int((chars.year_built > 2026).sum()))
sources.year_band(chars.year_built).value_counts().sort_index()

buildings with characteristics 448,404
count   448,384.000
mean      1,934.372
std          33.373
min       1,836.000
25%       1,910.000
50%       1,926.000
75%       1,955.000
max       2,026.000


raw buildings with year built < 1830 (set to missing): 4 | earliest raw year: 1801 | after 2026: 0


year_built
<1900       68337
1900-29    177694
1930-49     45624
1950-69    102913
1970-85     10607
1986-99     14902
2000+       28307
Name: count, dtype: int64

## 4. Small Illinois systems (format tests)

In [12]:
rows = []
for name, loader in [("Burnham IL0310360", sources.burnham), ("East Dundee IL0890250", sources.east_dundee)]:
    d = loader()
    status = line_status(d.utility_side_material, d.customer_side_material)
    rows.append({
        "system": name, "lines": len(d),
        "utility_unknown": d.utility_side_material.eq("Unknown").mean(),
        "customer_unknown": d.customer_side_material.eq("Unknown").mean(),
        "lead_or_grr": status.isin(["Lead", "GRR"]).mean(),
        "line_unknown": status.eq("Unknown").mean(),
        "with_pin": d.pin.notna().mean(), "with_year_built": d.year_built.notna().mean(),
        "with_diameter": d.diameter_in.notna().mean(),
        "duplicate_address": d.address.str.upper().duplicated(keep=False).mean(),
    })
pd.DataFrame(rows).set_index("system").T

system,Burnham IL0310360,East Dundee IL0890250
lines,"1,208.000","1,497.000"
utility_unknown,0.342,0.087
customer_unknown,0.076,0.087
lead_or_grr,0.573,0.171
line_unknown,0.306,0.087
with_pin,0.999,0.790
with_year_built,0.000,0.957
with_diameter,0.000,0.057
duplicate_address,0.000,0.069


In [13]:
ed = sources.east_dundee()
print("East Dundee rows flagged Duplicate by the village:", int(ed.duplicate_flag.sum()))
print("derived vs reported EntireMatlClass agreement:",
      (line_status(ed.utility_side_material, ed.customer_side_material) == ed.reported_line_class).mean())

East Dundee rows flagged Duplicate by the village: 343
derived vs reported EntireMatlClass agreement: 0.9993319973279893
